# semiotic-emergence-gpu

GPU-accelerated evolutionary communication simulation.

**Runtime setup:** Go to Runtime > Change runtime type > select **T4 GPU** before running.

In [ ]:
# 1. Install
!git clone https://github.com/onblueroses/semiotic-emergence-gpu.git
%cd semiotic-emergence-gpu
!pip install -q -e '.[dev]'
!pip install -q --upgrade 'jax[cuda12-local]'

In [ ]:
# 2. Verify GPU
import jax
print(f"Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
assert jax.default_backend() == "gpu", "No GPU detected! Change runtime type."

In [ ]:
# 3. Run tests
!python -m pytest tests/ -q

In [ ]:
# 4. Rust-equivalent validation: 384 pop, 56 grid, 500 ticks, 100 gens
import os, time
os.makedirs("runs/rust-equiv", exist_ok=True)
os.chdir("runs/rust-equiv")

start = time.time()
!python -m semgpu.main 42 100 --pop 384 --grid 56 --ticks 500
elapsed = time.time() - start
print(f"\nTotal: {elapsed:.1f}s ({elapsed/100:.2f}s/gen)")

os.chdir("/content/semiotic-emergence-gpu")

In [ ]:
# 5. Inspect results
import pandas as pd

df = pd.read_csv("runs/rust-equiv/output.csv")
print(f"Columns: {len(df.columns)}")
print(f"Rows: {len(df)}")
print()
print("First 5 gens:")
print(df[["generation", "avg_fitness", "max_fitness", "signals_emitted",
          "mutual_info", "iconicity", "zone_deaths", "signal_entropy"]].head())
print()
print("Last 5 gens:")
print(df[["generation", "avg_fitness", "max_fitness", "signals_emitted",
          "mutual_info", "iconicity", "zone_deaths", "signal_entropy"]].tail())

In [ ]:
# 6. Plot fitness and key metrics
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].plot(df["generation"], df["avg_fitness"], label="avg")
axes[0, 0].plot(df["generation"], df["max_fitness"], label="max", alpha=0.5)
axes[0, 0].set_title("Fitness")
axes[0, 0].legend()

axes[0, 1].plot(df["generation"], df["mutual_info"])
axes[0, 1].set_title("Mutual Information")

axes[0, 2].plot(df["generation"], df["iconicity"])
axes[0, 2].set_title("Iconicity")

axes[1, 0].plot(df["generation"], df["zone_deaths"])
axes[1, 0].set_title("Zone Deaths")

axes[1, 1].plot(df["generation"], df["signal_entropy"])
axes[1, 1].set_title("Signal Entropy")

axes[1, 2].plot(df["generation"], df["jsd_no_pred"], label="no pred")
axes[1, 2].plot(df["generation"], df["jsd_pred"], label="pred", alpha=0.7)
axes[1, 2].set_title("Receiver JSD")
axes[1, 2].legend()

for ax in axes.flat:
    ax.set_xlabel("Generation")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 7. Scale test: 1k prey
os.makedirs("runs/scale-1k", exist_ok=True)
os.chdir("runs/scale-1k")

start = time.time()
!python -m semgpu.main 0 50 --pop 1000 --grid 80 --food 300 --ticks 200
elapsed = time.time() - start
print(f"\nTotal: {elapsed:.1f}s ({elapsed/50:.2f}s/gen)")

os.chdir("/content/semiotic-emergence-gpu")

In [ ]:
# 8. VRAM usage
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader

In [ ]:
# 9. Scale test: 5k prey (may OOM - that's useful data)
os.makedirs("runs/scale-5k", exist_ok=True)
os.chdir("runs/scale-5k")

start = time.time()
try:
    !python -m semgpu.main 0 10 --pop 5000 --grid 150 --food 1000 --ticks 100
    elapsed = time.time() - start
    print(f"\nTotal: {elapsed:.1f}s ({elapsed/10:.2f}s/gen)")
except Exception as e:
    print(f"Failed (expected at scale): {e}")

os.chdir("/content/semiotic-emergence-gpu")

In [ ]:
# 10. Download results
!ls -la runs/*/output.csv runs/*/trajectory.csv runs/*/input_mi.csv 2>/dev/null

# Uncomment to download:
# from google.colab import files
# !tar czf results.tar.gz runs/
# files.download('results.tar.gz')